# Class-balancing tissue composition at low atlas-cell counts

Reproduces the numbers used in the Reviewer 1 Round 2 response (Section 4.2):
how many unique tissues are present in the 100k-cell blood+atlas source corpora
before class balancing, especially at **1** and **10** atlas cells.

## Data used

| Role | Path |
|------|------|
| Atlas cohort | `data/sctab/BaseModel_scTabAll_seed0_allbase_TrainingData.h5ad` |
| Blood base | `data/sctab/BaseModel_scTabBloodOnly_seed0_bloodbase_TrainingData.h5ad` |

Sampling matches `scripts/balance_data.py` → `read_and_preprocess_data_for_scvi_sctab`:
- total corpus size $N = 100{,}000$
- draw `atlas_count` cells (without replacement) from the atlas file
- draw `100000 - atlas_count` cells from the blood file
- seeds `{42, 43, 44, 45, 46}` (same as training / Supplementary Fig. 8)

## Important: which obs label?

- The **training bash script** `bash/run_balance_data_scvi.sh` sets
  `OBS_CLASS_LABEL=("tissue")` — so the **released class-balancing runs used `tissue`**
  (fine-grained; 146 unique labels in the full atlas file).
- The atlas file also has **`tissue_general`** (42 unique broad tissue types).

This notebook reports both, so you can check which matches the manuscript / Supplementary Table 1 wording.

In [1]:
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from utils import set_seed  # noqa: E402

ATLAS_PATH = REPO_ROOT / "data/sctab/BaseModel_scTabAll_seed0_allbase_TrainingData.h5ad"
BLOOD_PATH = REPO_ROOT / "data/sctab/BaseModel_scTabBloodOnly_seed0_bloodbase_TrainingData.h5ad"

ALL_TRAIN_CELLS = 100_000
SEEDS = [42, 43, 44, 45, 46]
ATLAS_COUNTS = [0, 1, 10, 100, 1000, 10000, 50000]
LABEL_COLS = ["tissue_general", "tissue"]

OUT_DIR = REPO_ROOT / "eval_scripts/outputs/class_balancing_tissue_composition"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("Atlas exists:", ATLAS_PATH.exists(), ATLAS_PATH)
print("Blood exists:", BLOOD_PATH.exists(), BLOOD_PATH)

REPO_ROOT: /Users/zeinab/Documents/MSR_internship/project/sc-AR-github-repo/sc-AR
Atlas exists: True /Users/zeinab/Documents/MSR_internship/project/sc-AR-github-repo/sc-AR/data/sctab/BaseModel_scTabAll_seed0_allbase_TrainingData.h5ad
Blood exists: True /Users/zeinab/Documents/MSR_internship/project/sc-AR-github-repo/sc-AR/data/sctab/BaseModel_scTabBloodOnly_seed0_bloodbase_TrainingData.h5ad


## 1. Load source corpora and report label vocabulary sizes

In [2]:
atlas = ad.read_h5ad(ATLAS_PATH)
blood = ad.read_h5ad(BLOOD_PATH)

print(f"Atlas shape: {atlas.shape}")
print(f"Blood shape: {blood.shape}")
print()

vocab = []
for col in LABEL_COLS:
    n_atlas = atlas.obs[col].nunique()
    n_blood = blood.obs[col].nunique()
    vocab.append(
        {
            "label_col": col,
            "n_unique_in_atlas": n_atlas,
            "n_unique_in_blood": n_blood,
            "blood_labels": sorted(blood.obs[col].astype(str).unique().tolist()),
        }
    )
    print(f"{col}:")
    print(f"  unique in atlas: {n_atlas}")
    print(f"  unique in blood: {n_blood}")
    print(f"  blood labels: {sorted(blood.obs[col].astype(str).unique().tolist())}")

vocab_df = pd.DataFrame(vocab)
vocab_df.to_csv(OUT_DIR / "label_vocabulary_sizes.csv", index=False)
vocab_df

/Users/zeinab/opt/anaconda3/envs/zeinab-env/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/Users/zeinab/opt/anaconda3/envs/zeinab-env/lib/python3.10/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Atlas shape: (101879, 12023)
Blood shape: (101879, 12023)

tissue_general:
  unique in atlas: 42
  unique in blood: 1
  blood labels: ['blood']
tissue:
  unique in atlas: 146
  unique in blood: 3
  blood labels: ['blood', 'umbilical cord blood', 'venous blood']


,label_col,n_unique_in_atlas,n_unique_in_blood,blood_labels
0,tissue_general,42,1,[blood]
1,tissue,146,3,"[blood, umbilical cord blood, venous blood]"


## 2. Simulate blood+atlas source corpora (same sampling as `balance_data.py`)

For each `(atlas_count, seed)`, count:
- unique labels in the **atlas draw only**
- unique labels in the **combined 100k corpus** (blood + atlas)

Mean ± std are over the five seeds.

In [3]:
# Work on obs tables only (much faster than AnnData.copy()).
atlas_obs = atlas.obs[LABEL_COLS].copy()
blood_obs = blood.obs[LABEL_COLS].copy()
for col in LABEL_COLS:
    atlas_obs[col] = atlas_obs[col].astype(str)
    blood_obs[col] = blood_obs[col].astype(str)


def sample_indices(atlas_count: int, seed: int):
    """Match scripts/balance_data.py::read_and_preprocess_data_for_scvi_sctab indexing."""
    set_seed(seed)
    atlas_idx = np.random.choice(len(atlas_obs), atlas_count, replace=False)
    set_seed(seed)
    blood_count = ALL_TRAIN_CELLS - atlas_count
    blood_idx = np.random.choice(len(blood_obs), blood_count, replace=False)
    assert len(atlas_idx) + len(blood_idx) == ALL_TRAIN_CELLS
    return blood_idx, atlas_idx


rows = []
for atlas_count in ATLAS_COUNTS:
    for seed in SEEDS:
        blood_idx, atlas_idx = sample_indices(atlas_count, seed)
        for col in LABEL_COLS:
            atlas_labels = set(atlas_obs.iloc[atlas_idx][col].tolist()) if atlas_count > 0 else set()
            blood_labels = set(blood_obs.iloc[blood_idx][col].tolist())
            combined = blood_labels | atlas_labels
            rows.append(
                {
                    "atlas_count": atlas_count,
                    "seed": seed,
                    "label_col": col,
                    "n_unique_atlas_only": len(atlas_labels),
                    "n_unique_blood_only": len(blood_labels),
                    "n_unique_combined": len(combined),
                    "atlas_labels": ";".join(sorted(atlas_labels)),
                }
            )

per_seed_df = pd.DataFrame(rows)
per_seed_df.to_csv(OUT_DIR / "unique_labels_per_seed.csv", index=False)
print("Wrote", OUT_DIR / "unique_labels_per_seed.csv")
per_seed_df.head(20)

Wrote /Users/zeinab/Documents/MSR_internship/project/sc-AR-github-repo/sc-AR/eval_scripts/outputs/class_balancing_tissue_composition/unique_labels_per_seed.csv


,atlas_count,seed,label_col,n_unique_atlas_only,n_unique_blood_only,n_unique_combined,atlas_labels
0,0,42,tissue_general,0,1,1,
1,0,42,tissue,0,3,3,
2,0,43,tissue_general,0,1,1,
3,0,43,tissue,0,3,3,
4,0,44,tissue_general,0,1,1,
5,0,44,tissue,0,3,3,
6,0,45,tissue_general,0,1,1,
7,0,45,tissue,0,3,3,
8,0,46,tissue_general,0,1,1,
9,0,46,tissue,0,3,3,


## 3. Summary: mean ± std across seeds (main table for the response)

In [4]:
summary = (
    per_seed_df.groupby(["label_col", "atlas_count"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        atlas_unique_mean=("n_unique_atlas_only", "mean"),
        atlas_unique_std=("n_unique_atlas_only", "std"),
        atlas_unique_min=("n_unique_atlas_only", "min"),
        atlas_unique_max=("n_unique_atlas_only", "max"),
        combined_unique_mean=("n_unique_combined", "mean"),
        combined_unique_std=("n_unique_combined", "std"),
        combined_unique_min=("n_unique_combined", "min"),
        combined_unique_max=("n_unique_combined", "max"),
    )
)

# round for readability
for c in summary.columns:
    if summary[c].dtype.kind == "f":
        summary[c] = summary[c].round(2)

summary.to_csv(OUT_DIR / "unique_labels_summary_mean_std.csv", index=False)
print("Wrote", OUT_DIR / "unique_labels_summary_mean_std.csv")
summary

Wrote /Users/zeinab/Documents/MSR_internship/project/sc-AR-github-repo/sc-AR/eval_scripts/outputs/class_balancing_tissue_composition/unique_labels_summary_mean_std.csv


,label_col,atlas_count,n_seeds,atlas_unique_mean,atlas_unique_std,atlas_unique_min,atlas_unique_max,combined_unique_mean,combined_unique_std,combined_unique_min,combined_unique_max
0,tissue,0,5,0.0,0.00,0,0,3.0,0.00,3,3
1,tissue,1,5,1.0,0.00,1,1,3.4,0.55,3,4
2,tissue,10,5,6.6,1.14,5,8,8.6,1.14,7,10
3,tissue,100,5,30.4,1.67,29,33,32.4,1.67,31,35
4,tissue,1000,5,80.2,7.53,74,91,81.2,7.56,74,92
5,tissue,10000,5,126.6,4.10,123,132,127.0,4.24,124,133
6,tissue,50000,5,142.0,1.22,141,144,142.0,1.22,141,144
7,tissue_general,0,5,0.0,0.00,0,0,1.0,0.00,1,1
8,tissue_general,1,5,1.0,0.00,1,1,1.4,0.55,1,2
9,tissue_general,10,5,4.8,0.84,4,6,4.8,0.84,4,6


### Focus: 1 and 10 atlas cells (numbers cited in the response)

Previous draft response used **`tissue_general`** (42 atlas tissues):
- 1 atlas cell → mean **1.0** unique atlas tissue
- 10 atlas cells → mean **4.8** unique atlas tissues (range 4–6)

If the manuscript / Supp. Table 1 refers to fine-grained **`tissue`**, use that column instead.

In [5]:
focus = summary[summary["atlas_count"].isin([1, 10])].copy()
focus["atlas_unique_mean_pm_std"] = focus.apply(
    lambda r: f"{r['atlas_unique_mean']:.1f} ± {r['atlas_unique_std']:.1f}",
    axis=1,
)
focus["atlas_unique_range"] = focus.apply(
    lambda r: f"{int(r['atlas_unique_min'])}–{int(r['atlas_unique_max'])}",
    axis=1,
)
focus[
    [
        "label_col",
        "atlas_count",
        "atlas_unique_mean_pm_std",
        "atlas_unique_range",
        "combined_unique_mean",
        "combined_unique_std",
    ]
]

,label_col,atlas_count,atlas_unique_mean_pm_std,atlas_unique_range,combined_unique_mean,combined_unique_std
1,tissue,1,1.0 ± 0.0,1–1,3.4,0.55
2,tissue,10,6.6 ± 1.1,5–8,8.6,1.14
8,tissue_general,1,1.0 ± 0.0,1–1,1.4,0.55
9,tissue_general,10,4.8 ± 0.8,4–6,4.8,0.84


## 4. Implied per-tissue replication under class balancing

Class balancing (`balance_data_class_balancing`) targets
`sample_count_per_class = int(N / T)` cells per unique label present in the
**combined** corpus, with replacement when a class is smaller than the target.

After the 90/10 split, training size ≈ `0.9 * sample_count_per_class` per class.

**Note:** the bash script balances on `tissue`, not `tissue_general`.
Replication counts below are therefore most accurate for `label_col='tissue'`.

In [6]:
def implied_replication(n_unique_combined: int, atlas_count: int, n_unique_atlas: int):
    """Rough implied cells-per-class before the 90/10 split."""
    if n_unique_combined == 0:
        return np.nan
    target = ALL_TRAIN_CELLS // n_unique_combined
    train_target = int(0.9 * target)
    # mean original atlas cells per captured atlas label (very rough)
    mean_orig = (atlas_count / n_unique_atlas) if n_unique_atlas > 0 else np.nan
    fold = (target / mean_orig) if mean_orig and mean_orig > 0 else np.nan
    return target, train_target, mean_orig, fold


repl_rows = []
for _, r in per_seed_df.iterrows():
    if r["atlas_count"] not in (1, 10):
        continue
    target, train_target, mean_orig, fold = implied_replication(
        r["n_unique_combined"], r["atlas_count"], r["n_unique_atlas_only"]
    )
    repl_rows.append(
        {
            **r.to_dict(),
            "target_cells_per_class_presplit": target,
            "approx_train_cells_per_class": train_target,
            "mean_original_atlas_cells_per_captured_label": mean_orig,
            "approx_replication_fold": fold,
        }
    )

repl_df = pd.DataFrame(repl_rows)
repl_summary = (
    repl_df.groupby(["label_col", "atlas_count"], as_index=False)
    .agg(
        target_presplit_mean=("target_cells_per_class_presplit", "mean"),
        train_per_class_mean=("approx_train_cells_per_class", "mean"),
        mean_orig_per_label=("mean_original_atlas_cells_per_captured_label", "mean"),
        replication_fold_mean=("approx_replication_fold", "mean"),
        replication_fold_std=("approx_replication_fold", "std"),
    )
    .round(1)
)
repl_summary.to_csv(OUT_DIR / "implied_replication_at_1_and_10.csv", index=False)
repl_summary

,label_col,atlas_count,target_presplit_mean,train_per_class_mean,mean_orig_per_label,replication_fold_mean,replication_fold_std
0,tissue,1,29999.8,26999.4,1.0,29999.8,4564.2
1,tissue,10,11801.4,10620.8,1.6,7639.6,329.6
2,tissue_general,1,80000.0,72000.0,1.0,80000.0,27386.1
3,tissue_general,10,21333.2,19199.8,2.1,9999.9,0.2


## 5. Optional: validate against saved class-balanced `.h5ad` files

If local sample files exist (e.g. under the manuscript `sample_shared_data` folder),
compare observed unique tissue counts after balancing. Paths can be edited below.

In [7]:
SAMPLE_FILES = [
    {
        "atlas_count": 0,
        "seed": 42,
        "path": Path(
            "/Users/zeinab/Documents/MSR_internship/manuscript/nature_communications/"
            "rebuttal/sample_shared_data/sample_data/"
            "bloodbase_0_atlas_seed42_2000HVGs_class_balancing_train_adata.h5ad"
        ),
    },
    {
        "atlas_count": 10000,
        "seed": 43,
        "path": Path(
            "/Users/zeinab/Documents/MSR_internship/manuscript/nature_communications/"
            "rebuttal/sample_shared_data/sample_data/"
            "bloodbase_10000_atlas_seed43_2000HVGs_class_balancing_train_adata.h5ad"
        ),
    },
]

val_rows = []
for item in SAMPLE_FILES:
    p = item["path"]
    if not p.exists():
        print("Missing:", p)
        continue
    a = ad.read_h5ad(p, backed="r")
    row = {
        "file": p.name,
        "atlas_count": item["atlas_count"],
        "seed": item["seed"],
        "n_cells": a.n_obs,
        "n_unique_tissue": a.obs["tissue"].nunique(),
        "n_unique_tissue_general": a.obs["tissue_general"].nunique(),
    }
    val_rows.append(row)
    print(row)

if val_rows:
    val_df = pd.DataFrame(val_rows)
    val_df.to_csv(OUT_DIR / "saved_balanced_files_unique_labels.csv", index=False)
    display(val_df)
else:
    print("No saved balanced files found for validation.")

/Users/zeinab/opt/anaconda3/envs/zeinab-env/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


{'file': 'bloodbase_0_atlas_seed42_2000HVGs_class_balancing_train_adata.h5ad', 'atlas_count': 0, 'seed': 42, 'n_cells': 90000, 'n_unique_tissue': 3, 'n_unique_tissue_general': 1}
{'file': 'bloodbase_10000_atlas_seed43_2000HVGs_class_balancing_train_adata.h5ad', 'atlas_count': 10000, 'seed': 43, 'n_cells': 90000, 'n_unique_tissue': 124, 'n_unique_tissue_general': 41}


/Users/zeinab/opt/anaconda3/envs/zeinab-env/lib/python3.10/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


,file,atlas_count,seed,n_cells,n_unique_tissue,n_unique_tissue_general
0,bloodbase_0_atlas_seed42_2000HVGs_class_balanc...,0,42,90000,3,1
1,bloodbase_10000_atlas_seed43_2000HVGs_class_ba...,10000,43,90000,124,41


## Takeaway for the response letter

1. **Numbers were computed by re-sampling** the two source `.h5ad` files above with seeds 42–46 — **not** by reading every saved balanced training file (most of those live on the cluster).
2. Use **`tissue_general`** if you want the “42 tissues” framing; use **`tissue`** if you want numbers consistent with the actual `OBS_CLASS_LABEL` used in `run_balance_data_scvi.sh`.
3. CSVs are written to `eval_scripts/outputs/class_balancing_tissue_composition/` for easy copy into the letter if the reviewer asks.